# 신규 크레인 35장 파인튜닝임
검수 완료 데이터 ZIP과 기존 균열·부식 가중치를 사용함. T4 GPU 런타임 사용이 필요함. 기존 학습 출력과 분리된 실행임.

In [ ]:
!pip -q install segmentation-models-pytorch==0.5.0
import torch
assert torch.cuda.is_available(), '런타임을 GPU로 연결해야 함'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
ROOT=Path('/content/crane35_0926')
ROOT.mkdir(exist_ok=True)
# 준비 ZIP, 검수 완료 dataset ZIP, 기존 부식 결과 ZIP 업로드 단계임
uploaded=files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        target=ROOT/Path(name).stem
        target.mkdir(exist_ok=False)
        with zipfile.ZipFile(name) as z:
            assert all(not Path(x).is_absolute() and '..' not in Path(x).parts for x in z.namelist())
            z.extractall(target)
print([str(x) for x in ROOT.rglob('dataset.json')])
print([str(x) for x in ROOT.rglob('*.pt')])

In [ ]:
# 실제 업로드 경로에 맞춘 설정 단계임
SCRIPT=ROOT/'crane35_prepare/train_finetune.py'
DATA=ROOT/'crane35_dataset/dataset'
CRACK=ROOT/'crane35_prepare/best_E5_tversky.pt'
CORROSION=ROOT/'corrosion_base_results0926/best_corrosion_efficientnet-b0.pt'
RUN=ROOT/'runs'
for p in [SCRIPT,DATA/'dataset.json',CRACK,CORROSION]:
    assert p.is_file(), f'파일 확인 필요: {p}'

In [ ]:
import subprocess,sys
for task,weights in [('crack',CRACK),('corrosion',CORROSION)]:
    subprocess.run([sys.executable,'-u',str(SCRIPT),'--task',task,'--data',str(DATA),'--weights',str(weights),'--out',str(RUN/task)],check=True)

In [ ]:
import json,shutil
for task in ['crack','corrosion']:
    r=json.loads((RUN/task/'results.json').read_text())
    assert r['status']=='completed'
    print(task,r['delta'],'검토후보',r['review_candidate'])
archive=shutil.make_archive('/content/crane35_finetune_results0926','zip',root_dir=ROOT)
files.download(archive)